In [2]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load saved knowledge base chunks
with open("../api/knowledge_base/chunks.json", "r") as f:
    knowledge_base = json.load(f)

# Load saved embeddings
embeddings = np.load("../api/knowledge_base/embeddings.npy")

print(f"Loaded {len(knowledge_base)} chunks")
print(f"Embedding shape: {embeddings.shape}")

# FAISS requires float32 -- ensure correct dtype
# Some systems save as float64 by default which FAISS rejects
embeddings = embeddings.astype(np.float32)

# Get the embedding dimension -- 384 for our model
dimension = embeddings.shape[1]
print(f"Embedding dimension: {dimension}")

# Build a FAISS index
# IndexFlatIP uses Inner Product similarity
# For normalised vectors inner product equals cosine similarity
# so we get the same ranking as before but via FAISS infrastructure
index = faiss.IndexFlatIP(dimension)

# Normalise embeddings before adding to index
# This converts dot product search into cosine similarity search
faiss.normalize_L2(embeddings)

# Add all embeddings to the index
index.add(embeddings)

print(f"\nFAISS index built successfully.")
print(f"Total vectors in index: {index.ntotal}")

# Save the FAISS index to disk
faiss.write_index(index, "../api/knowledge_base/faiss_index.bin")
print("FAISS index saved to disk.")

Loaded 9 chunks
Embedding shape: (9, 384)
Embedding dimension: 384

FAISS index built successfully.
Total vectors in index: 9
FAISS index saved to disk.


In [3]:
# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def faiss_search(question, top_k=3):
    # Embed the question
    question_embedding = embedding_model.encode([question])
    question_embedding = question_embedding.astype(np.float32)

    # Normalise query vector -- must match how index vectors were normalised
    faiss.normalize_L2(question_embedding)

    # Search the index
    # D = distances (similarity scores), I = indices of top results
    # top_k tells FAISS how many nearest neighbours to return
    D, I = index.search(question_embedding, top_k)

    # Build results
    results = []
    for score, idx in zip(D[0], I[0]):
        results.append({
            "topic": knowledge_base[idx]["topic"],
            "text": knowledge_base[idx]["text"],
            "similarity": round(float(score), 4)
        })
    return results

# Test with the same questions we used before
test_questions = [
    "which region is underperforming",
    "which category has the worst profit margin",
    "are there any dangerous transactions"
]

for question in test_questions:
    print(f"Question: {question}")
    results = faiss_search(question)
    for r in results:
        print(f"  [{r['similarity']}] {r['topic']}")
    print()

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 7128.11it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Question: which region is underperforming
  [0.5939] region_West
  [0.5251] region_East
  [0.5238] region_South

Question: which category has the worst profit margin
  [0.7253] category_Technology
  [0.6425] category_Furniture
  [0.5548] region_Central

Question: are there any dangerous transactions
  [0.4464] worst_transaction
  [0.2503] overall_summary
  [0.1679] region_Central



In [4]:
# Load the saved index from disk -- exactly as the API will do
loaded_index = faiss.read_index("../api/knowledge_base/faiss_index.bin")

print(f"Loaded index contains {loaded_index.ntotal} vectors")

# Run a test search using the loaded index
question = "which region has the most risk anomalies"
question_embedding = embedding_model.encode([question])
question_embedding = question_embedding.astype(np.float32)
faiss.normalize_L2(question_embedding)

D, I = loaded_index.search(question_embedding, 3)

print(f"\nTest query: {question}")
print("\nRetrieved chunks:")
for score, idx in zip(D[0], I[0]):
    print(f"  [{round(float(score), 4)}] {knowledge_base[idx]['topic']}")

Loaded index contains 9 vectors

Test query: which region has the most risk anomalies

Retrieved chunks:
  [0.5846] worst_transaction
  [0.464] region_West
  [0.4217] region_South
